# 🧠 SPaRG-MF: Spiking Max-Former — Colab Training

This notebook trains the **SPaRG-MF** (Spiking, Precision-Aware, Routed, and Gated Max-Former) on **ImageNet-1K**.

### Prerequisites
1. **Runtime**: Go to `Runtime → Change runtime type → GPU` (T4 or better).
2. **Dataset**: You need the ImageNet-1K dataset accessible. This notebook supports:
   - Loading from Google Drive (if you've uploaded it)
   - Downloading via the Kaggle API
3. **Code**: The SPaRG-MF repository should be in your Google Drive.

---

## 1. Mount Google Drive & Navigate to Project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ─── EDIT THIS PATH to match where SPaRG-MF lives in YOUR Drive ───
PROJECT_DIR = '/content/drive/MyDrive/SPaRG-MF'

import os
os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')
!ls -la

## 2. Install Dependencies

In [ ]:
# Install all project dependencies
!pip install -q timm>=0.9.0 spikingjelly>=0.0.0.0.14 einops

# Verify GPU is available
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 3. Verify Model Builds Correctly

Quick sanity check: build the model and run a dummy forward pass to catch any import or shape errors before committing to a long training run.

In [ ]:
import sys
sys.path.insert(0, PROJECT_DIR)

from models.maxformer_snn import SpikingMaxFormer
from spikingjelly.activation_based import functional

# Build a small config to verify everything works
test_model = SpikingMaxFormer(
    img_size=224,
    in_channels=3,
    num_classes=1000,
    embed_dims=384,
    depths=[1, 2, 7],
    num_heads=8,
    time_steps=2,  # small T for quick test
    enable_head_gate=True,
    enable_token_gate=False,
    enable_mixed_prec=False,
).cuda()

total_params = sum(p.numel() for p in test_model.parameters())
print(f'✅ Model built successfully! Parameters: {total_params / 1e6:.2f} M')

# Quick forward pass
dummy = torch.randn(2, 3, 224, 224).cuda()
with torch.no_grad():
    out = test_model(dummy)
    functional.reset_net(test_model)
print(f'✅ Forward pass OK! Input: {dummy.shape} → Output: {out.shape}')

del test_model, dummy, out
torch.cuda.empty_cache()
print('✅ Cleanup done. Ready to train!')

## 4. Dataset Setup

Choose **one** of the options below depending on how your dataset is stored.

### Option A: ImageNet from Google Drive
If you have ImageNet-1K already in your Drive, just set the path:
```python
DATA_DIR = '/content/drive/MyDrive/datasets/imagenet'
```

### Option B: Download via Kaggle API
Upload your `kaggle.json` to Colab, then run the cell below.

### Option C: Use CIFAR-100 (quick alternative)
If you don't have ImageNet, use the built-in CIFAR-100 cell further below.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Option A: Set this to your ImageNet path in Google Drive
# ═══════════════════════════════════════════════════════════════
# DATA_DIR = '/content/drive/MyDrive/datasets/imagenet'

# ═══════════════════════════════════════════════════════════════
#  Option B: Download ImageNet via Kaggle API
#  (Uncomment and run — requires kaggle.json uploaded to Colab)
# ═══════════════════════════════════════════════════════════════
# !pip install -q kaggle
# !mkdir -p ~/.kaggle
# # Upload your kaggle.json first via Colab file upload
# from google.colab import files
# uploaded = files.upload()  # upload kaggle.json
# !mv kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json
# !kaggle competitions download -c imagenet-object-localization-challenge -p /content/imagenet
# !cd /content/imagenet && unzip -q '*.zip'
# DATA_DIR = '/content/imagenet/ILSVRC/Data/CLS-LOC'

# ═══════════════════════════════════════════════════════════════
#  For now, default to a placeholder — CHANGE THIS!
# ═══════════════════════════════════════════════════════════════
DATA_DIR = '/content/drive/MyDrive/datasets/imagenet'
print(f'Dataset directory: {DATA_DIR}')

if os.path.isdir(DATA_DIR):
    print('✅ Dataset directory found!')
    for split in ['train', 'val', 'validation']:
        sp = os.path.join(DATA_DIR, split)
        if os.path.isdir(sp):
            n_classes = len([d for d in os.listdir(sp) if os.path.isdir(os.path.join(sp, d))])
            print(f'   {split}/: {n_classes} class folders')
else:
    print('⚠️  Dataset directory NOT found. Please update DATA_DIR above.')

## 5. Configure Training Hyperparameters

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Training Configuration — Edit these as needed
# ═══════════════════════════════════════════════════════════════

CONFIG = {
    # Model
    'model': 'sparg_mf_10_384',     # Options: sparg_mf_10_384, sparg_mf_10_512, sparg_mf_10_768
    'time_steps': 4,
    
    # Training
    'batch_size': 32,               # Reduce if you get OOM errors (T4: 16-32, A100: 64)
    'epochs': 100,
    'lr': 5e-4,
    'weight_decay': 0.05,
    'workers': 4,                   # Colab usually has 2 CPUs, 4 is safe with prefetch
    
    # Augmentation
    'mixup': 0.8,
    'cutmix': 1.0,
    'smoothing': 0.1,
    
    # Scheduler
    'sched': 'cosine',
    'min_lr': 1e-6,
    'warmup_epochs': 5,
    'warmup_lr': 1e-6,
    
    # EMA
    'model_ema': True,
    'model_ema_decay': 0.9998,
    
    # Checkpoint
    'resume': '',                   # Set to 'output/checkpoint.pth' to resume
    'output_dir': '/content/drive/MyDrive/SPaRG-MF/output',  # Save to Drive!
}

print('Training Configuration:')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')

## 6. 🚀 Launch Training

This runs `train_imagenet.py` directly in single-GPU mode (appropriate for Colab). Checkpoints are saved to your Google Drive so they survive runtime disconnects.

In [ ]:
# Build the command
cmd = f"""python train_imagenet.py \
    --model {CONFIG['model']} \
    --time_steps {CONFIG['time_steps']} \
    --batch_size {CONFIG['batch_size']} \
    --epochs {CONFIG['epochs']} \
    --lr {CONFIG['lr']} \
    --weight_decay {CONFIG['weight_decay']} \
    --workers {CONFIG['workers']} \
    --mixup {CONFIG['mixup']} \
    --cutmix {CONFIG['cutmix']} \
    --smoothing {CONFIG['smoothing']} \
    --sched {CONFIG['sched']} \
    --min_lr {CONFIG['min_lr']} \
    --warmup_epochs {CONFIG['warmup_epochs']} \
    --warmup_lr {CONFIG['warmup_lr']} \
    --data_dir {DATA_DIR} \
    --output_dir {CONFIG['output_dir']}"""

if CONFIG['model_ema']:
    cmd += f" --model-ema --model-ema-decay {CONFIG['model_ema_decay']}"

if CONFIG['resume']:
    cmd += f" --resume {CONFIG['resume']}"

print('='*60)
print('Running command:')
print(cmd)
print('='*60)

!{cmd}

## 7. Resume Training After Disconnect

If Colab disconnects, simply re-run cells 1-5 and then run this cell instead of cell 6:

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Resume from last saved checkpoint
# ═══════════════════════════════════════════════════════════════
RESUME_PATH = f"{CONFIG['output_dir']}/checkpoint.pth"

if os.path.isfile(RESUME_PATH):
    ckpt = torch.load(RESUME_PATH, map_location='cpu')
    print(f'✅ Found checkpoint at epoch {ckpt["epoch"]} with acc {ckpt.get("acc1", "?"):.2f}%')
    print(f'   Best acc so far: {ckpt.get("best_acc1", "?"):.2f}%')
    del ckpt
    
    cmd_resume = cmd + f" --resume {RESUME_PATH}"
    print(f'\nResuming training...')
    !{cmd_resume}
else:
    print(f'⚠️  No checkpoint found at {RESUME_PATH}')
    print('   Run cell 6 to start training from scratch.')

## 8. Evaluate Best Model

In [ ]:
import sys
sys.path.insert(0, PROJECT_DIR)

import torch
from timm.models import create_model
from timm.data import create_dataset, create_loader
from spikingjelly.activation_based import functional
import models.maxformer_snn  # register models

BEST_CKPT = f"{CONFIG['output_dir']}/model_best.pth"

if not os.path.isfile(BEST_CKPT):
    print(f'⚠️  No best model found at {BEST_CKPT}')
else:
    # Load best model
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    model = create_model(
        CONFIG['model'],
        pretrained=False,
        num_classes=1000,
        time_steps=CONFIG['time_steps'],
    )
    
    ckpt = torch.load(BEST_CKPT, map_location='cpu')
    
    # Load EMA weights if available, otherwise base weights
    if 'model_ema' in ckpt:
        model.load_state_dict(ckpt['model_ema'])
        print('Loaded EMA weights from best checkpoint')
    else:
        model.load_state_dict(ckpt['state_dict'])
        print('Loaded base weights from best checkpoint')
    
    print(f"Checkpoint epoch: {ckpt['epoch']} | Saved acc: {ckpt.get('acc1', '?')}%")
    model.to(device).eval()
    
    # Evaluate on validation set
    dataset_eval = create_dataset('imagenet', root=DATA_DIR, split='validation', is_training=False)
    eval_loader = create_loader(
        dataset_eval,
        input_size=(3, 224, 224),
        batch_size=64,
        is_training=False,
        use_prefetcher=True,
        num_workers=4,
        crop_pct=0.875,
    )
    
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, targets in eval_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            functional.reset_net(model)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    
    final_acc = 100. * correct / total
    print(f'\n🏆 Final Validation Accuracy: {final_acc:.2f}%')
    
    # Print spike statistics
    stats = model.count_spikes()
    if stats:
        print('\n📊 Homeostatic Spike Rates:')
        for k, v in stats.items():
            print(f"  {k}: rate={v['avg_spike_rate']:.4f}, threshold={v['v_threshold']:.4f}")

---

## Alternative: CIFAR-100 Quick Training

If you don't have ImageNet, use this cell to train on CIFAR-100 as a quick validation run. This uses the built-in `train_cifar10.py` logic adapted for CIFAR-100.

In [ ]:
import sys
sys.path.insert(0, PROJECT_DIR)

import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from spikingjelly.activation_based import functional
from models.maxformer_snn import SpikingMaxFormer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {device}')

# ─── Hyperparameters ───
BATCH_SIZE = 64
EPOCHS = 50
LR = 1e-3
NUM_CLASSES = 100  # CIFAR-100

# ─── Data ───
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
])

train_set = datasets.CIFAR100(root='./data', train=True, download=True, transform=transform_train)
test_set = datasets.CIFAR100(root='./data', train=False, download=True, transform=transform_test)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# ─── Model ───
model = SpikingMaxFormer(
    img_size=32,
    in_channels=3,
    num_classes=NUM_CLASSES,
    embed_dims=128,
    depths=[1, 1, 3],
    num_heads=4,
    time_steps=4,
    enable_head_gate=True,
    enable_token_gate=False,
    enable_mixed_prec=False,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model Parameters: {total_params / 1e6:.2f} M')

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_acc = 0.0
for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss, correct, total = 0.0, 0, 0
    t0 = time.time()
    
    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        functional.reset_net(model)
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
    
    scheduler.step()
    train_acc = 100. * correct / total
    
    # Eval
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            functional.reset_net(model)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    test_acc = 100. * correct / total
    
    marker = ' ★' if test_acc > best_acc else ''
    best_acc = max(best_acc, test_acc)
    print(f'Epoch {epoch:3d}/{EPOCHS} | {time.time()-t0:.0f}s | '
          f'Train: {train_acc:.1f}% | Test: {test_acc:.1f}% | Best: {best_acc:.1f}%{marker}')
    
    # Spike stats (every 10 epochs)
    if epoch % 10 == 0:
        stats = model.count_spikes()
        if stats:
            rates = [f"{k.split('.')[-1]}={v['avg_spike_rate']:.3f}" for k, v in list(stats.items())[:4]]
            print(f'  📊 Spike rates: {" | ".join(rates)}')

print(f'\n🏆 CIFAR-100 Training Complete. Best Accuracy: {best_acc:.2f}%')